# DisjointSet & Strongly Connected Components — Test Notebook

This notebook exercises two independent parts of `algokit-graph`:

1. **`Graph.strongly_connected_components()`** — Kosaraju's algorithm on directed graphs (cells 1–10).
2. **`algokit.DisjointSet`** — the standalone union–find data structure (cells 11–20).

Each test cell is self-contained: it builds a small graph or `DisjointSet` by hand, runs one
API call, and prints the result so the expected output is visible directly under the cell.
Run the cells top to bottom after the setup step below.


## Setup

Clones the `algokit-graph` repository and `cd`s into it, so the next cell can install it
directly from the local source tree.

In [ ]:
!git clone https://github.com/ayush09062004/algokit-graph.git
%cd algokit-graph

Cloning into 'algokit-graph'...
remote: Enumerating objects: 854, done.
remote: Counting objects: 100% (40/40), done.
remote: Compressing objects: 100% (26/26), done.
remote: Total 854 (delta 16), reused 24 (delta 9), pack-reused 814 (from 1)
Receiving objects: 100% (854/854), 52.72 MiB | 11.64 MiB/s, done.
Resolving deltas: 100% (438/438), done.
/content/algokit-graph


Installs the package in editable mode (`pip install -e .`). This compiles the
C++ extension via the `pybind11`/CMake build defined in `pyproject.toml` — expect this
step to take a minute or two the first time.

In [ ]:
!pip install -e .

Obtaining file:///content/algokit-graph
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for algokit (pyproject.toml) ... done
  Created wheel for algokit: filename=algokit-0.1.0-cp312-cp312-linux_x86_64.whl size=168003 sha256=800c485c24e5807b835d0d8ddd1298439da23b2361beed727d7160f2ec33db1c
  Stored in directory: /tmp/pip-ephem-wheel-cache-d21frkkw/wheels/6f/cc/a1/7c6a46c6b115b53184cea0dc45b2d81e45921650c6221c31c8
Successfully built algokit


Sanity-check the install: import `algokit` and print its version and module
docstring. If this cell fails, the build in the previous cell didn't succeed.

In [ ]:
import algokit

print("Version:", algokit.__version__)
print(algokit.__doc__)

Version: 1.0.0
AlgoKit Python Bindings


### SCC — a single 4-cycle

Build a directed 4-cycle `0 → 1 → 2 → 3 → 0`. Every vertex can reach every other vertex,
so the whole graph is one strongly connected component.

**Expect:** `component_count() == 1`, `components() == [[0, 1, 2, 3]]`.

In [ ]:
import algokit

g = algokit.Graph.directed(4)

g.add_edges([
    (0,1),
    (1,2),
    (2,3),
    (3,0)
])

scc = g.strongly_connected_components()

print(scc)
print("Count:", scc.component_count())
print("IDs:", scc.component_ids())
print("Components:", scc.components())

StronglyConnectedComponentsResult(component_count=1)
Count: 1
IDs: [0, 0, 0, 0]
Components: [[0, 3, 2, 1]]


### SCC — no edges at all

A directed graph with 5 vertices and zero edges. With no edges, no vertex can reach any
other, so every vertex is its own SCC.

**Expect:** `component_count() == 5`, each component a singleton `[v]`.

In [ ]:
g = algokit.Graph.directed(5)

scc = g.strongly_connected_components()

print(scc.component_count())
print(scc.component_ids())
print(scc.components())

5
[4, 3, 2, 1, 0]
[[4], [3], [2], [1], [0]]


### SCC — multiple components joined by one-way bridges

An 8-vertex graph containing three separate cycles — `{0,1,2}`, `{3,4,5}`, `{6,7}` — linked
by one-directional bridge edges (`2→3` and `6→5`) that don't create a path back. Since the
bridges only go one way, they don't merge the cycles into a single SCC.

**Expect:** `component_count() == 3`, with `{0,1,2}`, `{3,4,5}`, `{6,7}` as the three groups
(order within `components()` may vary).

In [ ]:
g = algokit.Graph.directed(8)

g.add_edges([
    (0,1),
    (1,2),
    (2,0),

    (2,3),

    (3,4),
    (4,5),
    (5,3),

    (6,5),
    (6,7),
    (7,6)
])

scc = g.strongly_connected_components()

print("Count:", scc.component_count())
print("IDs:", scc.component_ids())
print("Components:", scc.components())

Count: 3
IDs: [1, 1, 1, 2, 2, 2, 0, 0]
Components: [[6, 7], [0, 2, 1], [3, 5, 4]]


### SCC — one directed edge, no cycle

Two vertices with a single edge `0 → 1`. There's no way back from `1` to `0`, so they can't
be strongly connected to each other.

**Expect:** `component_count() == 2` (two singletons).

In [ ]:
g = algokit.Graph.directed(2)

g.add_edge(0,1)

scc = g.strongly_connected_components()

print(scc.component_count())
print(scc.component_ids())
print(scc.components())

2
[0, 1]
[[0], [1]]


### SCC — two vertices, both directions

Same two vertices as above, but now with `0 → 1` **and** `1 → 0`. Now they can reach each
other, so they merge into one SCC.

**Expect:** `component_count() == 1`, `components() == [[0, 1]]`.

In [ ]:
g = algokit.Graph.directed(2)

g.add_edge(0,1)
g.add_edge(1,0)

scc = g.strongly_connected_components()

print(scc.component_count())
print(scc.component_ids())
print(scc.components())

1
[0, 0]
[[0, 1]]


### SCC — self-loops don't merge distinct vertices

Three vertices; `0` and `1` each get a self-loop (`0→0`, `1→1`), and vertex `2` is isolated.
A self-loop makes a vertex trivially able to "reach itself", but it doesn't connect it to
any *other* vertex.

**Expect:** `component_count() == 3` — still three singleton components.

In [ ]:
g = algokit.Graph.directed(3)

g.add_edge(0,0)
g.add_edge(1,1)

scc = g.strongly_connected_components()

print(scc.component_count())
print(scc.component_ids())
print(scc.components())

3
[2, 1, 0]
[[2], [1], [0]]


Reuses the `scc` result from the self-loop graph above and prints
`component_id(v)` for every vertex — a quick way to eyeball the full id → component mapping
instead of reading the nested `components()` list.

In [ ]:
for v in range(g.vertex_count()):
    print(f"{v} -> {scc.component_id(v)}")

0 -> 2
1 -> 1
2 -> 0


### Error handling — out-of-range vertex

Calling `component_id()` with a vertex id (`100`) that doesn't exist in the graph should
raise, not silently return a bogus id.

**Expect:** an `IndexError` (surfaced from the C++ `std::out_of_range`).

In [ ]:
try:
    scc.component_id(100)
except Exception as e:
    print(type(e).__name__, e)

IndexError Vertex index is out of range.


## DisjointSet basics

Creates a fresh `DisjointSet` with 5 elements. Freshly constructed, every element is its
own singleton set and is its own `find()` root.

**Expect:** `component_count() == 5`, and `find(i) == i` for every `i`.

In [ ]:
import algokit

dsu = algokit.DisjointSet(5)

print(dsu.component_count())

for i in range(5):
    print(i, dsu.find(i))

5
0 0
1 1
2 2
3 3
4 4


### `unite()` merges two sets

Before uniting, `connected(0, 1)` is `False`. After `dsu.unite(0, 1)`, both elements share
the same root and `connected(0, 1)` becomes `True`; the total component count drops by one.

**Expect:** `False` → `True`, `find(0) == find(1)`, `component_count() == 4`.

In [ ]:
dsu = algokit.DisjointSet(5)

print(dsu.connected(0,1))

dsu.unite(0,1)

print(dsu.connected(0,1))

print(dsu.find(0))
print(dsu.find(1))

print(dsu.component_count())

False
True
0
0
4


### Chained unions

Unites `0–1`, `1–2`, `2–3` one at a time, joining all four into a single set while `4` and
`5` stay separate. Checks both a same-set pair (`0, 3`) and a different-set pair (`0, 5`).

**Expect:** `connected(0, 3) is True`, `connected(0, 5) is False`, `component_count() == 3`
(one set of size 4, plus `{4}` and `{5}`).

In [ ]:
dsu = algokit.DisjointSet(6)

dsu.unite(0,1)
dsu.unite(1,2)
dsu.unite(2,3)

print(dsu.connected(0,3))
print(dsu.connected(0,5))

for i in range(6):
    print(i, dsu.find(i))

print("Components:", dsu.component_count())

True
False
0 0
1 0
2 0
3 0
4 4
5 5
Components: 3


### `component_size()`

Builds two small groups — `{0,1,2}` via `unite(0,1)` + `unite(1,2)`, and `{4,5}` via a
single `unite(4,5)` — leaving `3` untouched. Checks that every member of a set reports the
same size, and that an untouched singleton reports size 1.

**Expect:** `component_size` is `3` for `0`, `1`, `2`; `2` for `4` and `5`; `1` for `3`.

In [ ]:
dsu = algokit.DisjointSet(8)

dsu.unite(0,1)
dsu.unite(1,2)

dsu.unite(4,5)

print(dsu.component_size(0))
print(dsu.component_size(1))
print(dsu.component_size(2))

print(dsu.component_size(4))
print(dsu.component_size(5))

print(dsu.component_size(3))

3
3
3
2
2
1


### `unite()` return value signals whether a merge happened

`unite()` returns `True` only the first time two different sets are merged. Calling it
again on already-connected elements — in either argument order — should return `False`
without changing anything further.

**Expect:** `True`, then `False`, then `False`; `component_count() == 2`.

In [ ]:
dsu = algokit.DisjointSet(3)

print(dsu.unite(0,1))
print(dsu.unite(0,1))
print(dsu.unite(1,0))

print(dsu.component_count())

True
False
False
2


### Uniting an element with itself

`unite(1, 1)` unions an element with itself — same root on both sides, so no merge happens.

**Expect:** `unite(1, 1)` returns `False` (or `True` depending on the exact same-root
handling — read the printed value), `connected(1, 1)` is `True`, and `component_count()`
is unchanged at `3`.

In [ ]:
dsu = algokit.DisjointSet(3)

print(dsu.unite(1,1))

print(dsu.connected(1,1))
print(dsu.component_count())

False
True
3


### Star union — everything joins one root

Unites every element `1..19` directly with element `0`, forming one large set from 20
singletons. Then asserts every element is connected back to `0`.

**Expect:** `component_count() == 1`; the assertion loop passes silently and prints
`"All connected!"`.

In [ ]:
n = 20

dsu = algokit.DisjointSet(n)

for i in range(1,n):
    dsu.unite(0,i)

print(dsu.component_count())

for i in range(n):
    assert dsu.connected(0,i)

print("All connected!")

1
All connected!


### Path compression sanity check

Chains 10 elements together (`0–1`, `1–2`, ..., `8–9`) and reads back `find(i)` for every
element. Because of path compression, every `find()` call after the first should return
the *same* root directly, regardless of how long the original union chain was.

**Expect:** every `find(i)` for `i` in `0..9` returns the same root value.

In [ ]:
dsu = algokit.DisjointSet(10)

for i in range(9):
    dsu.unite(i,i+1)

root = dsu.find(9)

for i in range(10):
    print(i, dsu.find(i))

print("Root:", root)

0 0
1 0
2 0
3 0
4 0
5 0
6 0
7 0
8 0
9 0
Root: 0


### Error handling — out-of-range elements

Runs `find`, `unite`, `connected`, and `component_size` with an out-of-range element id
(`100`) against a 5-element `DisjointSet`, and confirms each one raises rather than
corrupting internal state or returning a nonsense value.

**Expect:** four ✅ lines, one per method, each reporting the exception type and message.

In [ ]:
dsu = algokit.DisjointSet(5)

tests = [
    lambda: dsu.find(100),
    lambda: dsu.unite(0,100),
    lambda: dsu.connected(0,100),
    lambda: dsu.component_size(100),
]

for i, t in enumerate(tests, 1):
    try:
        t()
        print(f"❌ Test {i} failed")
    except Exception as e:
        print(f"✅ Test {i}: {type(e).__name__}: {e}")

✅ Test 1: IndexError: Vertex index is out of range.
✅ Test 2: IndexError: Vertex index is out of range.
✅ Test 3: IndexError: Vertex index is out of range.
✅ Test 4: IndexError: Vertex index is out of range.
